# Two-stage churn pipeline (time-based CV)

**Data:** `data/preprocessed/train` — same columns as released tables; multi-row tables are aggregated per `user_id` with counts / sums / existing categorical breakdowns only (no extra engineered signals).

**Target (3 classes):** `invol_churn`, `vol_churn`, `not_churned`.

**Stages:**
1. Binary: `not_churned` vs `churned` (invol + vol).
2. Binary on churned users: `vol_churn` vs `invol_churn`; at inference, combine probabilities:  
   \(P(\text{not})=p_1\), \(P(\text{vol})=(1-p_1)\cdot p_2\), \(P(\text{invol})=(1-p_1)\cdot(1-p_2)\).

**Validation:** outer `TimeSeriesSplit` on users sorted by `subscription_start_date`. **Hyperparameters:** optional `RandomizedSearchCV` with inner time-series CV (`TUNE_*` in imports cell).

**Metrics (0–100 scale):** **Composed** binary churn vs `not_churned` (from the two-stage probabilities). **Per stage:** stage 1 = churn vs stay (accuracy, F1 for churn, ROC-AUC); stage 2 = vol vs invol on validation users who churned only (accuracy, F1 for vol, ROC-AUC).

**Environment:** `/home/ansar/work/.venv`


In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
N_TS_SPLITS = 5
TUNE_HYPERPARAMS = True
TUNE_N_ITER = 12
TUNE_INNER_SPLITS = 3

DATA_DIR = Path("/home/ansar/work/hack-nu-26/data/preprocessed/train")
BEST_PARAMS_PATH = Path("/home/ansar/work/hack-nu-26/best_model_params.json")

# Binary churn vs not_churned uses y_binary_stay / composed P(churn) from the two-stage head.


## 1. Load tables and build one row per user

Join purchases ↔ attempts, aggregate generations from `DATA_DIR`. `train_users_transaction_attempts.csv` has no `user_id`; rows join via `transaction_id`.


In [2]:
def load_base_users() -> pd.DataFrame:
    u = pd.read_csv(
        DATA_DIR / "train_users.csv",
        usecols=[
            "user_id",
            "churn_status",
            "gdp_growth_pct",
            "log_churn_density",
            "gdp_per_capita",
        ],
    )
    u["gdp_growth_pct"] = pd.to_numeric(u["gdp_growth_pct"], errors="coerce")
    u["log_churn_density"] = pd.to_numeric(u["log_churn_density"], errors="coerce")
    u["gdp_per_capita"] = pd.to_numeric(u["gdp_per_capita"], errors="coerce")
    # Generation features from data/train/train_users.csv (e.g. feature_gen_delta_*, feature_log1p_total_gen)
    _gen_path = DATA_DIR.parent.parent / "train" / "train_users.csv"
    _gen = pd.read_csv(
        _gen_path,
        usecols=["user_id", "gen_delta_day1_minus_day14", "log1p_total_gen"],
        low_memory=False,
    ).drop_duplicates(subset=["user_id"], keep="first")
    u = u.merge(_gen, on="user_id", how="left")
    u["gen_delta_day1_minus_day14"] = pd.to_numeric(
        u["gen_delta_day1_minus_day14"], errors="coerce"
    )
    u["log1p_total_gen"] = pd.to_numeric(u["log1p_total_gen"], errors="coerce")
    prop = pd.read_csv(
        DATA_DIR / "train_users_properties.csv",
        usecols=["user_id", "subscription_start_date", "subscription_plan", "country_code"],
    )
    q = pd.read_csv(
        DATA_DIR / "train_users_quizzes.csv",
        usecols=[
            "user_id",
            "source",
            "team_size",
            "experience",
            "usage_plan",
            "frustration",
            "first_feature",
            "role",
        ],
    )
    df = u.merge(prop, on="user_id", how="left").merge(q, on="user_id", how="left")
    df["subscription_start_ts"] = pd.to_datetime(
        df["subscription_start_date"], utc=True, errors="coerce"
    ).astype("int64")
    return df.drop(columns=["subscription_start_date"])


def aggregate_purchases_attempts() -> pd.DataFrame:
    pur = pd.read_csv(
        DATA_DIR / "train_users_purchases.csv",
        usecols=["user_id", "transaction_id", "purchase_type", "purchase_amount_dollars"],
        low_memory=False,
    )
    ta_use = [
        "transaction_id",
        "amount_in_usd",
        "billing_address_country",
        "card_3d_secure_support",
        "card_country",
        "card_funding",
        "cvc_check",
        "digital_wallet",
        "is_3d_secure",
        "is_3d_secure_authenticated",
        "payment_method_type",
        "bank_country",
        "is_prepaid",
        "is_virtual",
        "is_business",
    ]
    ta = pd.read_csv(
        DATA_DIR / "train_users_transaction_attempts.csv",
        usecols=ta_use,
        low_memory=False,
    )
    for c in ta.columns:
        if c.startswith("is_"):
            ta[c] = ta[c].map(lambda x: str(x).lower() in {"true", "1"})
    m = pur.merge(ta, on="transaction_id", how="left", suffixes=("_pur", ""))
    bool_cols = [c for c in ta.columns if c.startswith("is_")]
    if bool_cols:
        m["att_card_flags_row"] = m[bool_cols].astype(np.float64).mean(axis=1)
    else:
        m["att_card_flags_row"] = 0.0
    mix_cols = [
        c
        for c in [
            "billing_address_country",
            "card_country",
            "card_funding",
            "payment_method_type",
            "cvc_check",
            "digital_wallet",
            "card_3d_secure_support",
            "bank_country",
        ]
        if c in m.columns
    ]
    if mix_cols:
        m["att_payment_mix_key"] = m[mix_cols].astype(str).apply(
            lambda r: "|".join(r.values), axis=1
        )
    else:
        m["att_payment_mix_key"] = ""
    agg_dict = {
        "transaction_id": "count",
        "purchase_amount_dollars": "sum",
        "purchase_type": "nunique",
        "amount_in_usd": "sum",
        "att_card_flags_row": "mean",
        "att_payment_mix_key": "nunique",
    }
    g = m.groupby("user_id", as_index=False).agg(agg_dict)
    return g.rename(
        columns={
            "transaction_id": "purch_n",
            "purchase_amount_dollars": "purch_amount_sum",
            "purchase_type": "purch_type_nunique",
            "amount_in_usd": "att_amount_sum",
            "att_card_flags_row": "att_card_flags_mean",
            "att_payment_mix_key": "att_payment_mix_nunique",
        }
    )


def aggregate_generations(chunksize: int = 2_000_000) -> pd.DataFrame:
    path = DATA_DIR / "train_users_generations.csv"
    usecols = ["user_id", "status", "generation_type"]
    status_parts: list[pd.Series] = []
    type_parts: list[pd.Series] = []
    for chunk in pd.read_csv(path, chunksize=chunksize, usecols=usecols):
        status_parts.append(chunk.groupby(["user_id", "status"]).size())
        type_parts.append(chunk.groupby(["user_id", "generation_type"]).size())
    st = pd.concat(status_parts).groupby(level=[0, 1]).sum().unstack(fill_value=0)
    st.columns = [f"gen_status_{c}" for c in st.columns.astype(str)]
    gt = pd.concat(type_parts).groupby(level=[0, 1]).sum().unstack(fill_value=0)
    gt.columns = [f"gen_type_{c}" for c in gt.columns.astype(str)]
    out = st.join(gt, how="outer").fillna(0).astype(np.float32)
    out["gen_total"] = out[[c for c in out.columns if c.startswith("gen_status_")]].sum(axis=1)
    return out.reset_index()


df = load_base_users()
pa = aggregate_purchases_attempts()
df = df.merge(pa, on="user_id", how="left")
gen = aggregate_generations()
df = df.merge(gen, on="user_id", how="left")

num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in df.columns if c not in num_cols and c not in {"user_id", "churn_status"}]
for c in cat_cols:
    df[c] = df[c].fillna("skipped").astype(str)

df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
df = df.sort_values("subscription_start_ts").reset_index(drop=True)

print(df.shape, "numeric:", len(num_cols), "categorical:", len(cat_cols))
df[["user_id", "churn_status", "subscription_start_ts"]].head()



(89389, 53) numeric: 42 categorical: 9


,user_id,churn_status,subscription_start_ts
0,user_975491cf-14bc-454c-a7fa-9d2e1b39f29b,vol_churn,-28475452793000000
1,user_fdf76ee7-1450-4a5b-ac90-3f4a00364b51,vol_churn,-28475452710000000
2,user_a059cddc-3534-441a-a10c-c689678b432c,invol_churn,-28475452684000000
3,user_ce8e9ea1-1090-4140-be5a-fd2d157a103e,not_churned,-28475452637000000
4,user_78e9fad9-9a19-4405-ae43-b2052373d68b,not_churned,-28475452535000000


## 2. Targets, matrices, preprocessing (OrdinalEncoder + imputation + optional scaling)

In [3]:
y_binary_stay = (df["churn_status"] == "not_churned").astype(np.int8).values  # 1 = not churned
y_churn_subtype = (df["churn_status"] == "vol_churn").astype(np.int8).values  # 1 = vol among all users

feature_cols = [c for c in df.columns if c not in {"user_id", "churn_status"}]
X_df = df[feature_cols].copy()

num_features = [c for c in feature_cols if c in num_cols]
cat_features = [c for c in feature_cols if c in cat_cols]

# Stage-specific feature sets (exclude leakage / stage-specific signals)
STAGE1_DROP = frozenset({"log_churn_density", "log1p_total_gen"})
STAGE2_DROP = frozenset({"gen_delta_day1_minus_day14"})
cols_stage1 = [c for c in feature_cols if c not in STAGE1_DROP]
cols_stage2 = [c for c in feature_cols if c not in STAGE2_DROP]
num_stage1 = [c for c in cols_stage1 if c in num_cols]
cat_stage1 = [c for c in cols_stage1 if c in cat_cols]
num_stage2 = [c for c in cols_stage2 if c in num_cols]
cat_stage2 = [c for c in cols_stage2 if c in cat_cols]

for c in cat_features:
    X_df[c] = X_df[c].fillna("skipped").astype(str)


def make_preprocessor(
    scale: bool,
    *,
    num_cols_arg: list[str] | None = None,
    cat_cols_arg: list[str] | None = None,
) -> ColumnTransformer:
    nf = num_features if num_cols_arg is None else num_cols_arg
    cf = cat_features if cat_cols_arg is None else cat_cols_arg
    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
        + ([("scaler", StandardScaler())] if scale else [])
    )
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, nf),
            ("cat", cat_pipe, cf),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def hierarchical_class_proba(p_stay: np.ndarray, p_vol_if_churn: np.ndarray) -> np.ndarray:
    """p_stay = P(not_churned); p_vol_if_churn = P(vol_churn | churned). Columns: invol, vol, not."""
    p_stay = np.clip(p_stay, 1e-8, 1 - 1e-8)
    p_vol_if_churn = np.clip(p_vol_if_churn, 1e-8, 1 - 1e-8)
    p_churn = 1.0 - p_stay
    p_vol = p_churn * p_vol_if_churn
    p_invol = p_churn * (1.0 - p_vol_if_churn)
    return np.column_stack([p_invol, p_vol, p_stay])


def metrics_binary_churn_pct(y_stay_val: np.ndarray, proba3: np.ndarray) -> dict[str, float]:
    """y_stay_val: 1 = not_churned, 0 = churned (vol+invol). proba3 columns: invol, vol, not."""
    y_churn = (y_stay_val == 0).astype(np.int8)
    p_churn = proba3[:, 0] + proba3[:, 1]
    pred_churn = (p_churn >= 0.5).astype(np.int8)
    return {
        "accuracy_pct": float(100.0 * accuracy_score(y_churn, pred_churn)),
        "f1_churn_pct": float(
            100.0 * f1_score(y_churn, pred_churn, pos_label=1, zero_division=0)
        ),
        "roc_auc_churn_pct": float(100.0 * roc_auc_score(y_churn, p_churn)),
    }

## 3. Time-based CV + model factories + temporal ensemble

Models: **XGBoost**, **CatBoost** (native categoricals), **RandomForest**, **voting** (0.5·XGB + 0.5·CatBoost on ordinal matrix). **Hyperparameters** tuned with `RandomizedSearchCV` + inner `TimeSeriesSplit` when `TUNE_HYPERPARAMS` is True (imports cell).

**Temporal ensemble:** Each fold trains **two** models — one on **all** training data, one on the **late half** only (more representative of the test-era distribution). Final predictions blend with `ENSEMBLE_ALPHA` (0.4 full + 0.6 late). This addresses temporal feature drift observed in `churn_importance_by_time_bin.ipynb`.

**Reporting:** Metrics are shown for the **last fold** first (most test-like), then last 2 folds, then all-fold average. The last fold trains on ~80% of data and validates on the most recent ~20% — the closest simulation of the actual train→test scenario.

For each fold and model, **`met`** = composed binary churn vs not (from the two-stage head). **`per_stage`** table = **stage 1** (churn vs stay: accuracy, F1 for churn, ROC-AUC for `P(churn)`) and **stage 2** on validation churned users only (vol vs invol: accuracy, F1 for vol, ROC-AUC for `P(vol)`). All in **0–100**.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier


def _binary_cls_metrics_pct(y: np.ndarray, p_pos: np.ndarray) -> tuple[float, float, float]:
    """y in 0/1, p_pos = P(y=1); threshold 0.5. Returns (accuracy, f1 for class 1, roc_auc) in 0-100."""
    y = np.asarray(y, dtype=np.int8)
    p = np.clip(np.asarray(p_pos, dtype=np.float64), 1e-8, 1.0 - 1e-8)
    pred = (p >= 0.5).astype(np.int8)
    acc = 100.0 * float(accuracy_score(y, pred))
    f1v = 100.0 * float(f1_score(y, pred, pos_label=1, zero_division=0))
    if np.unique(y).size < 2:
        auc = float("nan")
    else:
        auc = 100.0 * float(roc_auc_score(y, p))
    return acc, f1v, auc


def _median_param_dict(dicts: list[dict]) -> dict:
    """Aggregate RandomizedSearchCV best_params across time folds (median / rounded ints)."""
    if not dicts:
        return {}
    keys = dicts[0].keys()
    out: dict = {}
    for k in keys:
        vals = [float(d[k]) for d in dicts if k in d and d[k] is not None]
        if not vals:
            continue
        m = float(np.median(vals))
        if k in ("n_estimators", "max_depth", "min_samples_leaf", "iterations", "depth"):
            out[k] = int(round(m))
        else:
            out[k] = m
    return out


def load_best_params() -> dict:
    if not BEST_PARAMS_PATH.exists():
        return {}
    return json.loads(BEST_PARAMS_PATH.read_text(encoding="utf-8"))


def _jsonable(v):
    if isinstance(v, dict):
        return {k: _jsonable(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_jsonable(x) for x in v]
    if isinstance(v, np.generic):
        return v.item()
    return v


def merge_save_best_params(mode: str, payload: dict) -> None:
    data = load_best_params()
    data[mode] = _jsonable(payload)
    BEST_PARAMS_PATH.parent.mkdir(parents=True, exist_ok=True)
    BEST_PARAMS_PATH.write_text(json.dumps(data, indent=2), encoding="utf-8")
    print(f"Saved best params for mode={mode!r} -> {BEST_PARAMS_PATH}")


class CatBoostOrdinalEnsembleBlock(ClassifierMixin, BaseEstimator):
    """CatBoost on the ordinal-encoded design matrix (all numeric). Use :class:`Pool` so CatBoost
    does not infer cat/text columns from ndarray inputs (avoids CatBoostError in VotingClassifier).
    """

    def __init__(
        self,
        cat_column_indices,
        iterations: int = 400,
        depth: int = 6,
        learning_rate: float = 0.05,
        random_seed: int = 0,
    ):
        self.cat_column_indices = cat_column_indices
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.random_seed = random_seed

    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.estimator_type = "classifier"
        return tags

    def fit(self, X, y):
        X = np.ascontiguousarray(np.asarray(X, dtype=np.float64))
        y = np.asarray(y).astype(float).ravel()
        self.model_ = CatBoostClassifier(
            loss_function="Logloss",
            iterations=self.iterations,
            depth=self.depth,
            learning_rate=self.learning_rate,
            random_seed=self.random_seed,
            verbose=False,
            allow_writing_files=False,
        )
        self.model_.fit(Pool(X, label=y))
        return self

    def predict(self, X):
        X = np.ascontiguousarray(np.asarray(X, dtype=np.float64))
        return self.model_.predict(X)

    def predict_proba(self, X):
        X = np.ascontiguousarray(np.asarray(X, dtype=np.float64))
        return self.model_.predict_proba(X)


class CatBoostSklearnDF(ClassifierMixin, BaseEstimator):
    """Sklearn-compatible CatBoost on raw DataFrame with fixed cat column indices (for RandomizedSearchCV)."""

    def __init__(
        self,
        cat_feature_indices,
        iterations: int = 400,
        depth: int = 6,
        learning_rate: float = 0.05,
        random_seed: int = 0,
    ):
        self.cat_feature_indices = cat_feature_indices
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.random_seed = random_seed

    def fit(self, X, y):
        self.model_ = CatBoostClassifier(
            loss_function="Logloss",
            iterations=self.iterations,
            depth=self.depth,
            learning_rate=self.learning_rate,
            random_seed=self.random_seed,
            verbose=False,
            allow_writing_files=False,
        )
        self.model_.fit(X, y, cat_features=list(self.cat_feature_indices))
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


def _tune_xgb_binary(
    X: np.ndarray, y: np.ndarray, random_state: int, params_override: dict | None = None
) -> tuple[XGBClassifier, dict | None]:
    fixed_kw = dict(
        random_state=random_state, n_jobs=-1, eval_metric="logloss", verbosity=0
    )
    if params_override is not None and not TUNE_HYPERPARAMS:
        m = XGBClassifier(**params_override, **fixed_kw)
        return m.fit(X, y), dict(params_override)
    if not TUNE_HYPERPARAMS:
        m = XGBClassifier(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            **fixed_kw,
        )
        return m.fit(X, y), None
    base = XGBClassifier(**fixed_kw)
    dist = {
        "n_estimators": randint(200, 601),
        "max_depth": randint(4, 11),
        "learning_rate": loguniform(0.02, 0.12),
        "subsample": uniform(0.75, 0.24),
        "colsample_bytree": uniform(0.65, 0.34),
        "reg_lambda": loguniform(0.2, 5.0),
    }
    search = RandomizedSearchCV(
        base,
        param_distributions=dist,
        n_iter=TUNE_N_ITER,
        cv=TimeSeriesSplit(n_splits=TUNE_INNER_SPLITS),
        scoring="roc_auc",
        random_state=random_state,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X, y)
    return search.best_estimator_, dict(search.best_params_)


def _tune_rf_binary(
    X: np.ndarray, y: np.ndarray, random_state: int, params_override: dict | None = None
) -> tuple[RandomForestClassifier, dict | None]:
    fixed_kw = dict(random_state=random_state, n_jobs=-1)
    if params_override is not None and not TUNE_HYPERPARAMS:
        m = RandomForestClassifier(**params_override, **fixed_kw)
        return m.fit(X, y), dict(params_override)
    if not TUNE_HYPERPARAMS:
        m = RandomForestClassifier(
            n_estimators=300,
            max_depth=16,
            min_samples_leaf=4,
            **fixed_kw,
        )
        return m.fit(X, y), None
    base = RandomForestClassifier(**fixed_kw)
    dist = {
        "n_estimators": randint(150, 501),
        "max_depth": randint(8, 22),
        "min_samples_leaf": randint(1, 12),
    }
    search = RandomizedSearchCV(
        base,
        param_distributions=dist,
        n_iter=TUNE_N_ITER,
        cv=TimeSeriesSplit(n_splits=TUNE_INNER_SPLITS),
        scoring="roc_auc",
        random_state=random_state,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X, y)
    return search.best_estimator_, dict(search.best_params_)


def _tune_catboost_ordinal(
    X: np.ndarray,
    y: np.ndarray,
    cat_idx_ord: list[int],
    random_state: int,
    params_override: dict | None = None,
) -> tuple[CatBoostOrdinalEnsembleBlock, dict | None]:
    if params_override is not None and not TUNE_HYPERPARAMS:
        po = dict(params_override)
        est = CatBoostOrdinalEnsembleBlock(
            cat_column_indices=cat_idx_ord,
            iterations=int(po.get("iterations", 400)),
            depth=int(po.get("depth", 6)),
            learning_rate=float(po.get("learning_rate", 0.05)),
            random_seed=random_state,
        )
        return est.fit(X, y), po
    if not TUNE_HYPERPARAMS:
        est = CatBoostOrdinalEnsembleBlock(
            cat_column_indices=cat_idx_ord,
            iterations=400,
            depth=6,
            learning_rate=0.05,
            random_seed=random_state,
        )
        return est.fit(X, y), None
    base = CatBoostOrdinalEnsembleBlock(
        cat_column_indices=cat_idx_ord,
        iterations=400,
        depth=6,
        learning_rate=0.05,
        random_seed=random_state,
    )
    dist = {
        "iterations": randint(200, 601),
        "depth": randint(4, 10),
        "learning_rate": loguniform(0.02, 0.12),
    }
    search = RandomizedSearchCV(
        base,
        param_distributions=dist,
        n_iter=TUNE_N_ITER,
        cv=TimeSeriesSplit(n_splits=TUNE_INNER_SPLITS),
        scoring="roc_auc",
        random_state=random_state,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X, y)
    return search.best_estimator_, dict(search.best_params_)


def _tune_catboost_df(
    X_train: pd.DataFrame,
    y: np.ndarray,
    cat_idx: list[int],
    random_state: int,
    params_override: dict | None = None,
) -> tuple[CatBoostSklearnDF, dict | None]:
    if params_override is not None and not TUNE_HYPERPARAMS:
        po = dict(params_override)
        m = CatBoostSklearnDF(
            cat_feature_indices=cat_idx,
            iterations=int(po.get("iterations", 400)),
            depth=int(po.get("depth", 6)),
            learning_rate=float(po.get("learning_rate", 0.05)),
            random_seed=random_state,
        )
        return m.fit(X_train, y), po
    if not TUNE_HYPERPARAMS:
        m = CatBoostSklearnDF(
            cat_feature_indices=cat_idx,
            iterations=400,
            depth=6,
            learning_rate=0.05,
            random_seed=random_state,
        )
        return m.fit(X_train, y), None
    base = CatBoostSklearnDF(
        cat_feature_indices=cat_idx,
        iterations=400,
        depth=6,
        learning_rate=0.05,
        random_seed=random_state,
    )
    dist = {
        "iterations": randint(200, 601),
        "depth": randint(4, 10),
        "learning_rate": loguniform(0.02, 0.12),
    }
    search = RandomizedSearchCV(
        base,
        param_distributions=dist,
        n_iter=TUNE_N_ITER,
        cv=TimeSeriesSplit(n_splits=TUNE_INNER_SPLITS),
        scoring="roc_auc",
        random_state=random_state,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y)
    return search.best_estimator_, dict(search.best_params_)


def fit_predict_hierarchical(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    y_stay_train: np.ndarray,
    y_stay_val: np.ndarray,
    y_vol_train_all: np.ndarray,
    y_vol_val_all: np.ndarray,
    churned_train_mask: np.ndarray,
    mode: str,
) -> tuple[np.ndarray, dict[str, float], dict | None]:
    """Returns (val proba n×3, stage diagnostics, tune_meta or None)."""
    diag: dict[str, float] = {}
    tune_meta: dict | None = None

    stored = load_best_params() if not TUNE_HYPERPARAMS else {}

    if mode == "catboost":
        X1 = X_train[cols_stage1].copy()
        Xv1 = X_val[cols_stage1].copy()
        for c in cat_stage1:
            X1[c] = X1[c].fillna("skipped").astype(str).astype(object)
            Xv1[c] = Xv1[c].fillna("skipped").astype(str).astype(object)
        cat_idx = [X1.columns.get_loc(c) for c in cat_stage1]
        po1 = (stored.get("catboost") or {}).get("stage1")
        po2 = (stored.get("catboost") or {}).get("stage2")
        m1, p1 = _tune_catboost_df(X1, y_stay_train, cat_idx, RANDOM_STATE, po1)
        p_stay_val = m1.predict_proba(Xv1)[:, 1]

        X_tr2 = X_train.iloc[churned_train_mask][cols_stage2].copy()
        Xv2 = X_val[cols_stage2].copy()
        for c in cat_stage2:
            X_tr2[c] = X_tr2[c].fillna("skipped").astype(str).astype(object)
            Xv2[c] = Xv2[c].fillna("skipped").astype(str).astype(object)
        cat_idx2 = [X_tr2.columns.get_loc(c) for c in cat_stage2]
        y_tr2 = y_vol_train_all[churned_train_mask]
        m2, p2 = _tune_catboost_df(X_tr2, y_tr2, cat_idx2, RANDOM_STATE + 1, po2)
        p_vol_if_churn_val = m2.predict_proba(Xv2)[:, 1]
        if TUNE_HYPERPARAMS:
            tune_meta = {"stage1": p1, "stage2": p2}

    elif mode in {"xgb", "rf", "voting"}:
        pre1 = make_preprocessor(
            scale=False, num_cols_arg=num_stage1, cat_cols_arg=cat_stage1
        )
        Xtr1 = pre1.fit_transform(X_train[cols_stage1])
        Xva1 = pre1.transform(X_val[cols_stage1])

        pre2 = make_preprocessor(
            scale=False, num_cols_arg=num_stage2, cat_cols_arg=cat_stage2
        )
        pre2.fit(X_train[cols_stage2])
        Xtr2 = pre2.transform(X_train[cols_stage2])[churned_train_mask]
        Xva2 = pre2.transform(X_val[cols_stage2])

        if mode == "xgb":
            po1 = (stored.get("xgb") or {}).get("stage1")
            po2 = (stored.get("xgb") or {}).get("stage2")
            m1, p1 = _tune_xgb_binary(Xtr1, y_stay_train, RANDOM_STATE, po1)
            p_stay_val = m1.predict_proba(Xva1)[:, 1]
            m2, p2 = _tune_xgb_binary(
                Xtr2,
                y_vol_train_all[churned_train_mask],
                RANDOM_STATE + 1,
                po2,
            )
            p_vol_if_churn_val = m2.predict_proba(Xva2)[:, 1]
            if TUNE_HYPERPARAMS:
                tune_meta = {"stage1": p1, "stage2": p2}

        elif mode == "rf":
            po1 = (stored.get("rf") or {}).get("stage1")
            po2 = (stored.get("rf") or {}).get("stage2")
            m1, p1 = _tune_rf_binary(Xtr1, y_stay_train, RANDOM_STATE, po1)
            p_stay_val = m1.predict_proba(Xva1)[:, 1]
            m2, p2 = _tune_rf_binary(
                Xtr2,
                y_vol_train_all[churned_train_mask],
                RANDOM_STATE + 1,
                po2,
            )
            p_vol_if_churn_val = m2.predict_proba(Xva2)[:, 1]
            if TUNE_HYPERPARAMS:
                tune_meta = {"stage1": p1, "stage2": p2}

        else:  # voting = soft average of tuned XGB + CatBoost (ordinal matrix)
            n_num_s1 = len(num_stage1)
            cat_idx_ord_s1 = list(range(n_num_s1, n_num_s1 + len(cat_stage1)))
            n_num_s2 = len(num_stage2)
            cat_idx_ord_s2 = list(range(n_num_s2, n_num_s2 + len(cat_stage2)))
            sv = stored.get("voting") or {}
            xpo1 = (sv.get("xgb") or {}).get("stage1")
            xpo2 = (sv.get("xgb") or {}).get("stage2")
            cpo1 = (sv.get("catboost_ordinal") or {}).get("stage1")
            cpo2 = (sv.get("catboost_ordinal") or {}).get("stage2")
            x1, xp1 = _tune_xgb_binary(Xtr1, y_stay_train, RANDOM_STATE, xpo1)
            cb1, cp1 = _tune_catboost_ordinal(
                Xtr1, y_stay_train, cat_idx_ord_s1, RANDOM_STATE + 3, cpo1
            )
            p_stay_val = 0.5 * x1.predict_proba(Xva1)[:, 1] + 0.5 * cb1.predict_proba(Xva1)[
                :, 1
            ]
            x2, xp2 = _tune_xgb_binary(
                Xtr2,
                y_vol_train_all[churned_train_mask],
                RANDOM_STATE + 7,
                xpo2,
            )
            cb2, cp2 = _tune_catboost_ordinal(
                Xtr2,
                y_vol_train_all[churned_train_mask],
                cat_idx_ord_s2,
                RANDOM_STATE + 11,
                cpo2,
            )
            p_vol_if_churn_val = 0.5 * x2.predict_proba(Xva2)[:, 1] + 0.5 * cb2.predict_proba(
                Xva2
            )[:, 1]
            if TUNE_HYPERPARAMS:
                tune_meta = {
                    "xgb": {"stage1": xp1, "stage2": xp2},
                    "catboost_ordinal": {"stage1": cp1, "stage2": cp2},
                }
    else:
        raise ValueError(mode)

    y_churn_val = (y_stay_val == 0).astype(np.int8)
    p_churn_val = 1.0 - np.asarray(p_stay_val, dtype=np.float64)
    s1_acc, s1_f1, s1_auc = _binary_cls_metrics_pct(y_churn_val, p_churn_val)
    diag["stage1_accuracy_pct"] = s1_acc
    diag["stage1_f1_churn_pct"] = s1_f1
    diag["stage1_roc_auc_churn_pct"] = s1_auc
    mask_churn_val = y_stay_val == 0
    if mask_churn_val.sum() > 0:
        yv = y_vol_val_all[mask_churn_val].astype(np.int8)
        pv = p_vol_if_churn_val[mask_churn_val]
        s2_acc, s2_f1, s2_auc = _binary_cls_metrics_pct(yv, pv)
        diag["stage2_accuracy_pct"] = s2_acc
        diag["stage2_f1_vol_pct"] = s2_f1
        diag["stage2_roc_auc_vol_pct"] = s2_auc
    else:
        diag["stage2_accuracy_pct"] = float("nan")
        diag["stage2_f1_vol_pct"] = float("nan")
        diag["stage2_roc_auc_vol_pct"] = float("nan")

    proba3 = hierarchical_class_proba(p_stay_val, p_vol_if_churn_val)
    return proba3, diag, tune_meta


ENSEMBLE_ALPHA = 0.4  # weight for full-data model; (1 - alpha) for late-data model
LATE_FRAC = 0.3       # keep latest half of training data for the "late" model


def run_time_series_cv(mode: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    tss = TimeSeriesSplit(n_splits=N_TS_SPLITS)
    fold_rows = []
    diag_rows = []
    tune_stage1: list[dict] = []
    tune_stage2: list[dict] = []
    tune_v_x1: list[dict] = []
    tune_v_x2: list[dict] = []
    tune_v_c1: list[dict] = []
    tune_v_c2: list[dict] = []
    for fold, (tr, va) in enumerate(tss.split(X_df)):
        X_train, X_val = X_df.iloc[tr], X_df.iloc[va]
        ys_tr, ys_va = y_binary_stay[tr], y_binary_stay[va]
        yv_tr, yv_va = y_churn_subtype[tr], y_churn_subtype[va]
        churned_tr = df["churn_status"].iloc[tr].ne("not_churned").values

        # ── Full-data model ──
        proba_full, diag, tm = fit_predict_hierarchical(
            X_train, X_val, ys_tr, ys_va, yv_tr, yv_va, churned_tr, mode=mode,
        )

        # ── Late-data model (temporal ensemble) ──
        tr_late = tr[int(len(tr) * LATE_FRAC):]
        proba_late, _, _ = fit_predict_hierarchical(
            X_df.iloc[tr_late], X_val,
            y_binary_stay[tr_late], ys_va,
            y_churn_subtype[tr_late], yv_va,
            df["churn_status"].iloc[tr_late].ne("not_churned").values,
            mode=mode,
        )

        # ── Blend ──
        proba = ENSEMBLE_ALPHA * proba_full + (1 - ENSEMBLE_ALPHA) * proba_late
        if TUNE_HYPERPARAMS and tm:
            if mode == "voting":
                if tm.get("xgb", {}).get("stage1"):
                    tune_v_x1.append(tm["xgb"]["stage1"])
                if tm.get("xgb", {}).get("stage2"):
                    tune_v_x2.append(tm["xgb"]["stage2"])
                if tm.get("catboost_ordinal", {}).get("stage1"):
                    tune_v_c1.append(tm["catboost_ordinal"]["stage1"])
                if tm.get("catboost_ordinal", {}).get("stage2"):
                    tune_v_c2.append(tm["catboost_ordinal"]["stage2"])
            else:
                if tm.get("stage1"):
                    tune_stage1.append(tm["stage1"])
                if tm.get("stage2"):
                    tune_stage2.append(tm["stage2"])
        m = metrics_binary_churn_pct(ys_va, proba)
        m["fold"] = fold
        p_churn_blend = proba[:, 0] + proba[:, 1]
        y_churn_val = (ys_va == 0).astype(np.int8)
        pred_churn = (p_churn_blend >= 0.5).astype(np.int8)
        m["stage1_accuracy_pct"] = 100.0 * float(accuracy_score(y_churn_val, pred_churn))
        m["stage1_f1_churn_pct"] = 100.0 * float(f1_score(y_churn_val, pred_churn, pos_label=1, zero_division=0))
        m["stage1_roc_auc_churn_pct"] = 100.0 * float(roc_auc_score(y_churn_val, p_churn_blend))
        mask_c = ys_va == 0
        if mask_c.sum() > 0:
            p_vol_b = proba[mask_c, 1] / (proba[mask_c, 0] + proba[mask_c, 1]).clip(1e-8)
            yv_c = yv_va[mask_c].astype(np.int8)
            pred_v = (p_vol_b >= 0.5).astype(np.int8)
            m["stage2_accuracy_pct"] = 100.0 * float(accuracy_score(yv_c, pred_v))
            m["stage2_f1_vol_pct"] = 100.0 * float(f1_score(yv_c, pred_v, pos_label=1, zero_division=0))
            m["stage2_roc_auc_vol_pct"] = 100.0 * float(roc_auc_score(yv_c, p_vol_b)) if np.unique(yv_c).size >= 2 else float("nan")
        else:
            m["stage2_accuracy_pct"] = m["stage2_f1_vol_pct"] = m["stage2_roc_auc_vol_pct"] = float("nan")
        fold_rows.append(m)
        d = {"fold": fold, **{k: m[k] for k in m if k.startswith("stage")}}
        diag_rows.append(d)
    if TUNE_HYPERPARAMS:
        if mode == "voting":
            payload = {
                "xgb": {
                    "stage1": _median_param_dict(tune_v_x1),
                    "stage2": _median_param_dict(tune_v_x2),
                },
                "catboost_ordinal": {
                    "stage1": _median_param_dict(tune_v_c1),
                    "stage2": _median_param_dict(tune_v_c2),
                },
            }
            merge_save_best_params(mode, payload)
        elif tune_stage1 or tune_stage2:
            merge_save_best_params(
                mode,
                {
                    "stage1": _median_param_dict(tune_stage1),
                    "stage2": _median_param_dict(tune_stage2),
                },
            )
    return pd.DataFrame(fold_rows), pd.DataFrame(diag_rows)


MODELS = ["xgb", "catboost", "rf", "voting"]
summary = []
for name in MODELS:
    print("===", name, "===")
    met, diag = run_time_series_cv(name)
    met["model"] = name
    summary.append(met)
    display(met)
    per_stage = diag[
        [
            "fold",
            "stage1_accuracy_pct",
            "stage1_f1_churn_pct",
            "stage1_roc_auc_churn_pct",
            "stage2_accuracy_pct",
            "stage2_f1_vol_pct",
            "stage2_roc_auc_vol_pct",
        ]
    ]
    display(per_stage)

summary_df = pd.concat(summary, ignore_index=True)
_overall = ["accuracy_pct", "f1_churn_pct", "roc_auc_churn_pct"]
_stage1 = [
    "stage1_accuracy_pct",
    "stage1_f1_churn_pct",
    "stage1_roc_auc_churn_pct",
]
_stage2 = [
    "stage2_accuracy_pct",
    "stage2_f1_vol_pct",
    "stage2_roc_auc_vol_pct",
]
last_fold = N_TS_SPLITS - 1
last_df = summary_df[summary_df["fold"] == last_fold]
print(f"{'='*70}")
print(f"  LAST FOLD (fold {last_fold}) — most similar to test distribution")
print(f"{'='*70}")
display(last_df.groupby("model")[_overall + _stage1 + _stage2].mean().round(2))

last2_df = summary_df[summary_df["fold"] >= last_fold - 1]
print(f"\nLast 2 folds (folds {last_fold-1}–{last_fold}):")
display(last2_df.groupby("model")[_overall].mean().round(2))

print("\nAll-fold average (binary churn from hierarchical proba, 0–100)")
display(summary_df.groupby("model")[_overall].agg(["mean", "std"]))
print("Stage 1 — stay vs churn (all folds)")
display(summary_df.groupby("model")[_stage1].agg(["mean", "std"]))
print("Stage 2 — voluntary vs involuntary (all folds)")
display(summary_df.groupby("model")[_stage2].agg(["mean", "std"]))



=== xgb ===
Saved best params for mode='xgb' -> /home/ansar/work/hack-nu-26/best_model_params.json


,accuracy_pct,f1_churn_pct,roc_auc_churn_pct,fold,stage1_accuracy_pct,stage1_f1_churn_pct,stage1_roc_auc_churn_pct,stage2_accuracy_pct,stage2_f1_vol_pct,stage2_roc_auc_vol_pct,model
0,64.196536,60.483035,69.458318,0,64.196536,60.483035,69.458318,67.058506,60.344828,74.156749,xgb
1,61.800242,56.111668,66.719021,1,61.800242,56.111668,66.719021,61.690744,57.304711,69.303028,xgb
2,61.256545,63.532980,66.948339,2,61.256545,63.532980,66.948339,66.123164,67.222947,71.638027,xgb
3,59.524768,52.127660,65.147984,3,59.524768,52.127660,65.147984,66.051417,63.428490,71.003550,xgb
4,59.209290,55.167835,63.005002,4,59.209290,55.167835,63.005002,66.206236,67.405355,71.778632,xgb


,fold,stage1_accuracy_pct,stage1_f1_churn_pct,stage1_roc_auc_churn_pct,stage2_accuracy_pct,stage2_f1_vol_pct,stage2_roc_auc_vol_pct
0,0,64.196536,60.483035,69.458318,67.058506,60.344828,74.156749
1,1,61.800242,56.111668,66.719021,61.690744,57.304711,69.303028
2,2,61.256545,63.532980,66.948339,66.123164,67.222947,71.638027
3,3,59.524768,52.127660,65.147984,66.051417,63.428490,71.003550
4,4,59.209290,55.167835,63.005002,66.206236,67.405355,71.778632


=== catboost ===


## 4. Train on full data + inference on test → submission.csv

Train the two-stage hierarchical model on **all** training data (full + late temporal ensemble),
then predict on the test set. Pick a **separate model per stage**:

| Config | Stage | Task | Options |
|--------|-------|------|---------|
| `STAGE1_MODEL` | 1 | churn vs not_churned | `"xgb"`, `"catboost"`, `"rf"`, `"voting"` |
| `STAGE2_MODEL` | 2 | vol_churn vs invol_churn | `"xgb"`, `"catboost"`, `"rf"`, `"voting"` |

When both stages use the same model, it trains only once (no wasted compute).

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CONFIG — pick one model per stage
# ══════════════════════════════════════════════════════════════
STAGE1_MODEL = "catboost"   # churn vs not_churned: "xgb", "catboost", "rf", "voting"
STAGE2_MODEL = "xgb"        # vol  vs invol:        "xgb", "catboost", "rf", "voting"
SUBMISSION_PATH = Path("/home/ansar/work/hack-nu-26/submission.csv")

# ── Load test data (same pipeline as train) ──
TEST_DIR = Path("/home/ansar/work/hack-nu-26/data/preprocessed/test")

def load_test_data() -> pd.DataFrame:
    u = pd.read_csv(TEST_DIR / "test_users.csv",
        usecols=["user_id", "gdp_growth_pct", "log_churn_density"])
    u["gdp_growth_pct"] = pd.to_numeric(u["gdp_growth_pct"], errors="coerce")
    u["log_churn_density"] = pd.to_numeric(u["log_churn_density"], errors="coerce")

    _gen = pd.read_csv(TEST_DIR.parent.parent / "test" / "test_users.csv",
        usecols=["user_id", "gen_delta_day1_minus_day14", "log1p_total_gen"],
        low_memory=False).drop_duplicates(subset=["user_id"], keep="first")
    u = u.merge(_gen, on="user_id", how="left")
    u["gen_delta_day1_minus_day14"] = pd.to_numeric(u["gen_delta_day1_minus_day14"], errors="coerce")
    u["log1p_total_gen"] = pd.to_numeric(u["log1p_total_gen"], errors="coerce")

    prop = pd.read_csv(TEST_DIR / "test_users_properties.csv",
        usecols=["user_id", "subscription_start_date", "subscription_plan", "country_code"])
    q = pd.read_csv(TEST_DIR / "test_users_quizzes.csv",
        usecols=["user_id", "source", "team_size", "experience",
                 "usage_plan", "frustration", "first_feature", "role"])
    tdf = u.merge(prop, on="user_id", how="left").merge(q, on="user_id", how="left")
    tdf["subscription_start_ts"] = pd.to_datetime(
        tdf["subscription_start_date"], utc=True, errors="coerce").astype("int64")
    tdf = tdf.drop(columns=["subscription_start_date"])

    pur = pd.read_csv(TEST_DIR / "test_users_purchases.csv",
        usecols=["user_id", "transaction_id", "purchase_type", "purchase_amount_dollars"],
        low_memory=False)
    ta_use = ["transaction_id", "amount_in_usd", "billing_address_country",
        "card_3d_secure_support", "card_country", "card_funding", "cvc_check",
        "digital_wallet", "is_3d_secure", "is_3d_secure_authenticated",
        "payment_method_type", "bank_country", "is_prepaid", "is_virtual", "is_business"]
    ta = pd.read_csv(TEST_DIR / "test_users_transaction_attempts.csv",
        usecols=ta_use, low_memory=False)
    for c in ta.columns:
        if c.startswith("is_"):
            ta[c] = ta[c].map(lambda x: str(x).lower() in {"true", "1"})
    m = pur.merge(ta, on="transaction_id", how="left", suffixes=("_pur", ""))
    bool_cols = [c for c in ta.columns if c.startswith("is_")]
    m["att_card_flags_row"] = m[bool_cols].astype(np.float64).mean(axis=1) if bool_cols else 0.0
    mix_cols = [c for c in ["billing_address_country", "card_country", "card_funding",
        "payment_method_type", "cvc_check", "digital_wallet",
        "card_3d_secure_support", "bank_country"] if c in m.columns]
    m["att_payment_mix_key"] = (
        m[mix_cols].astype(str).apply(lambda r: "|".join(r.values), axis=1)
        if mix_cols else "")
    g = m.groupby("user_id", as_index=False).agg({
        "transaction_id": "count", "purchase_amount_dollars": "sum",
        "purchase_type": "nunique", "amount_in_usd": "sum",
        "att_card_flags_row": "mean", "att_payment_mix_key": "nunique",
    }).rename(columns={
        "transaction_id": "purch_n", "purchase_amount_dollars": "purch_amount_sum",
        "purchase_type": "purch_type_nunique", "amount_in_usd": "att_amount_sum",
        "att_card_flags_row": "att_card_flags_mean",
        "att_payment_mix_key": "att_payment_mix_nunique",
    })
    tdf = tdf.merge(g, on="user_id", how="left")

    gen_path = TEST_DIR / "test_users_generations.csv"
    sp, tp = [], []
    for chunk in pd.read_csv(gen_path, chunksize=2_000_000,
                             usecols=["user_id", "status", "generation_type"]):
        sp.append(chunk.groupby(["user_id", "status"]).size())
        tp.append(chunk.groupby(["user_id", "generation_type"]).size())
    st = pd.concat(sp).groupby(level=[0, 1]).sum().unstack(fill_value=0)
    st.columns = [f"gen_status_{c}" for c in st.columns.astype(str)]
    gt = pd.concat(tp).groupby(level=[0, 1]).sum().unstack(fill_value=0)
    gt.columns = [f"gen_type_{c}" for c in gt.columns.astype(str)]
    out = st.join(gt, how="outer").fillna(0).astype(np.float32)
    out["gen_total"] = out[[c for c in out.columns if c.startswith("gen_status_")]].sum(axis=1)
    tdf = tdf.merge(out.reset_index(), on="user_id", how="left")

    return tdf


def _decompose_proba3(proba3: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Extract per-stage probabilities from combined 3-class output.
    Returns (p_stay, p_vol_if_churn)."""
    p_stay = proba3[:, 2]
    p_churn_total = (proba3[:, 0] + proba3[:, 1]).clip(1e-8)
    p_vol_if_churn = proba3[:, 1] / p_churn_total
    return p_stay, p_vol_if_churn


def _run_one_model(X_train, X_test_aligned, ys_tr, yv_tr, churned_tr, mode):
    """Train both stages with `mode`, predict on test, return proba3."""
    dummy = np.zeros(len(X_test_aligned), dtype=np.int8)
    proba3, _, _ = fit_predict_hierarchical(
        X_train, X_test_aligned, ys_tr, dummy, yv_tr, dummy, churned_tr, mode=mode,
    )
    return proba3


print("Loading test data...")
test_df = load_test_data()

test_num = [c for c in test_df.columns if pd.api.types.is_numeric_dtype(test_df[c])]
test_cat = [c for c in test_df.columns if c not in test_num and c not in {"user_id"}]
for c in test_cat:
    test_df[c] = test_df[c].fillna("skipped").astype(str)
test_df[test_num] = test_df[test_num].replace([np.inf, -np.inf], np.nan)

for c in feature_cols:
    if c not in test_df.columns:
        test_df[c] = 0 if c in num_cols else "skipped"
X_test = test_df[feature_cols].copy()
for c in cat_features:
    X_test[c] = X_test[c].fillna("skipped").astype(str)

test_user_ids = test_df["user_id"].values
print(f"Test: {test_df.shape}, aligned features: {X_test.shape}")

# ── Prepare data slices ──
churned_all = (df["churn_status"] != "not_churned").values
late_start = int(len(X_df) * LATE_FRAC)
X_late = X_df.iloc[late_start:]
ys_late = y_binary_stay[late_start:]
yv_late = y_churn_subtype[late_start:]
churned_late = churned_all[late_start:]

same_model = STAGE1_MODEL == STAGE2_MODEL
models_needed = [STAGE1_MODEL] if same_model else [STAGE1_MODEL, STAGE2_MODEL]

print(f"\nStage 1 model: {STAGE1_MODEL}")
print(f"Stage 2 model: {STAGE2_MODEL}")
print(f"Temporal ensemble: alpha={ENSEMBLE_ALPHA} (full) / {1-ENSEMBLE_ALPHA} (late)")
print(f"Late data: {len(X_late)} rows (last {100*(1-LATE_FRAC):.0f}% of train)\n")

# ── Train & predict (full + late) for each unique model ──
full_proba3 = {}
late_proba3 = {}
for mode in models_needed:
    print(f"  [{mode}] training on full data ({len(X_df)} rows)...")
    full_proba3[mode] = _run_one_model(
        X_df, X_test, y_binary_stay, y_churn_subtype, churned_all, mode)

    print(f"  [{mode}] training on late data ({len(X_late)} rows)...")
    late_proba3[mode] = _run_one_model(
        X_late, X_test, ys_late, yv_late, churned_late, mode)
    print(f"  [{mode}] done.\n")

# ── Decompose, blend per-stage, recombine ──
p_stay_full, _   = _decompose_proba3(full_proba3[STAGE1_MODEL])
_,  p_vol_full   = _decompose_proba3(full_proba3[STAGE2_MODEL])
p_stay_late, _   = _decompose_proba3(late_proba3[STAGE1_MODEL])
_,  p_vol_late   = _decompose_proba3(late_proba3[STAGE2_MODEL])

p_stay_blend = ENSEMBLE_ALPHA * p_stay_full + (1 - ENSEMBLE_ALPHA) * p_stay_late
p_vol_blend  = ENSEMBLE_ALPHA * p_vol_full  + (1 - ENSEMBLE_ALPHA) * p_vol_late

final_proba = hierarchical_class_proba(p_stay_blend, p_vol_blend)

# ── Write submission ──
LABELS = np.array(["invol_churn", "vol_churn", "not_churned"])
pred_labels = LABELS[np.argmax(final_proba, axis=1)]

submission = pd.DataFrame({"user_id": test_user_ids, "churn_status": pred_labels})
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"{'='*60}")
print(f"  submission.csv → {SUBMISSION_PATH}")
print(f"{'='*60}")
print(f"  Rows:    {len(submission)}")
print(f"  Stage 1: {STAGE1_MODEL}")
print(f"  Stage 2: {STAGE2_MODEL}")
print(f"  Ensemble alpha: {ENSEMBLE_ALPHA}")
print(f"\n  Label distribution:")
print(submission["churn_status"].value_counts().to_string())
display(submission.head(10))